# 03 — Metric-learning embedder (Approach 2)

Drop the classifier head entirely. Train the embedder with TripletMarginLoss
(semi-hard mining) over labeled `items_train`. After training, encode every item
once → cluster directly on cosine of the learned 128-d embeddings.

**Why:** removes shortcut features (raw cosine inputs) and directly optimizes
embedding geometry. Inference becomes one-pass + KNN, no per-edge model call.

**Library:** `pytorch-metric-learning` for `MPerClassSampler`, `TripletMarginMiner`,
and `losses.TripletMarginLoss`.

**TODO:**
1. Filter items_train to labels with ≥ 2 items.
2. 80/20 label-disjoint split (same labels as baseline val if possible — for fair comparison).
3. Train. Save best to `phase2/models/metric_v2.pth`.
4. Encode all items → `phase2/embeddings/learned_embeddings.pt`.
5. Hand off to `04_cluster_and_eval.ipynb`.

> Reminder: this is **not** an autoencoder. There is no reconstruction loss.
> The encoder takes (img_emb, txt_emb, price, geo, color, dept) → 128-d, supervised
> by triplet labels.


In [ ]:
import sys, os
sys.path.insert(0, '/Users/matouskovar/FIT/adm-sp/src')

REPO_ROOT     = '/Users/matouskovar/FIT/adm-sp'
DATA_DIR      = f'{REPO_ROOT}/data'
ARTIFACTS_DIR = f'{REPO_ROOT}/artifacts'
PHASE2_DIR    = f'{REPO_ROOT}/phase2'

# Read-only artifacts
TEXT_EMB_PATH      = f'{ARTIFACTS_DIR}/embeddings/text_multilingual.pt'
CLIP_EMB_PATH      = f'{ARTIFACTS_DIR}/embeddings/clip_image.pt'   # baseline (vanilla CLIP)
PIPELINE_PATH      = f'{ARTIFACTS_DIR}/pipelines/preprocessing.pkl'
VOCAB_DIR          = f'{ARTIFACTS_DIR}/vocabularies'
BASELINE_MODEL     = f'{ARTIFACTS_DIR}/models/siamese_baseline.pth'   # F1 ~ 0.85 on val
HARDMINED_MODEL    = f'{ARTIFACTS_DIR}/models/siamese_hardmined.pth'  # F1 ~ 0.0004 (broken)

# Phase 2 outputs
FASHION_EMB_PATH   = f'{PHASE2_DIR}/embeddings/fashion_clip_image.pt'
LEARNED_EMB_PATH   = f'{PHASE2_DIR}/embeddings/learned_embeddings.pt'
SIAMESE_V2_PATH    = f'{PHASE2_DIR}/models/siamese_v2.pth'
METRIC_V2_PATH     = f'{PHASE2_DIR}/models/metric_v2.pth'

import torch
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# %pip install pytorch-metric-learning


In [ ]:
# TODO: dataset that yields (img_emb, txt_emb, price, geo, color_multihot, dept_multihot, label)
# label is the integer class for sampler. Reuse LightweightItemDataset from phase1/05.


In [ ]:
# TODO: encoder = SiameseEmbedder (reuse), input image dim = 512 (FashionCLIP).
# from pytorch_metric_learning import losses, miners, samplers
# sampler = samplers.MPerClassSampler(labels, m=4, batch_size=256)
# miner   = miners.TripletMarginMiner(margin=0.2, type_of_triplets='semihard')
# loss_fn = losses.TripletMarginLoss(margin=0.2)


In [ ]:
# TODO: training loop with miner + loss_fn over the sampler.
# Track validation: cosine of held-out same-label vs different-label pairs.
# Save best to METRIC_V2_PATH.


In [ ]:
# TODO: post-training, encode every item in items_train + items_phase_1 (+ phase_2 when ready)
# and save to LEARNED_EMB_PATH. This is what 04_cluster_and_eval consumes.
